In [2]:
# %cd "drive/MyDrive/Colab Notebooks/QNLPModelTraining"
import os
import re
import sys
import json
import pickle
import contextlib
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Dict

from qiskit_aer import AerSimulator
from pytket.extensions.qiskit.backends.aer import AerBackend
# from qiskit.providers.aer import AerSimulator
# from pytket.extensions.qiskit import AerBackend

from lambeq.backend.grammar import Diagram, Id
from lambeq import (
    QuantumTrainer,
    TketModel, Dataset,
    SPSAOptimizer,
    BinaryCrossEntropyLoss,
    AtomicType, IQPAnsatz,
    
)

from lambeq import (
    DepCCGParser,
    IQPAnsatz,
    AtomicType,
    RemoveCupsRewriter,
    TketModel,
    Dataset,
    UnifyCodomainRewriter,
    Rewriter,
    SimpleRewriteRule
)


/home/green/QNLPModelTraining/qnlp_3_10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
parser = DepCCGParser(model='elmo', device=0) # device=:  -1 == CPU | 0 == GPU | 1 == second GPU

In [12]:
def create_rewriter():
    # Rule to delete conjunction boxes (“and”, “but”) # just the wire, no box
    conj_rule = SimpleRewriteRule(cod=AtomicType.CONJUNCTION, template=Id(AtomicType.CONJUNCTION))

    rewriter = Rewriter([
        'determiner',
        'auxiliary', # potentially risky. jei tai nepasalina daug qubitu, tai ismest
        'connector',
        # 'coordination',
        'prepositional_phrase', 
        # 'subject_rel_pronoun', # They don't hurt performance, and they act as a safety
    ])
    
    # rewriter.add_rules(conj_rule)
    return rewriter

rewriter = create_rewriter()
remove_cups = RemoveCupsRewriter()
unify = UnifyCodomainRewriter(output_type=AtomicType.SENTENCE)
ansatz = IQPAnsatz(
    {
        AtomicType.NOUN: 1,
        AtomicType.SENTENCE: 1,
    },
    n_layers=1,
    n_single_qubit_params=3,
)

simulator = AerSimulator(
    method="statevector",
    device="GPU",
    precision="single",
    cuStateVec_enable=True,
)

backend = AerBackend(simulation_method="statevector")
backend._qiskit_backend = simulator

backend_config = {
    "backend": backend,
    "compilation": backend.default_compilation_pass(2),
    "shots": 2,
    # "shots": 8192,
    # "shots": 16384,
}

/home/green/QNLPModelTraining/qnlp_3_10/lib/python3.10/site-packages/pytket/extensions/qiskit/backends/aer.py:129: UserWarning: More than one backend with name 'aer_simulator' is available. Picking one.
  warnings.warn(


In [17]:
sentences = [
    "The company announced a new quantum computing platform.",
    "The weather was sunny during the event.",
    "The platform could improve language processing.",
    "The audience applauded at the end of the presentation.",
]
# sentences = ["Alice likes Bob.","Bob likes Alice.","Alice writes code.","Bob reads books.",]

labels = np.array([
    [0, 1],
    [1, 0],
    [0, 1],
    [1, 0],
])

raw_diagrams = parser.sentences2diagrams(
    sentences,
    suppress_exceptions=True
)

circuits, kept_sentences, kept_labels = [], [], []

for sentence, diagram, label in zip(sentences, raw_diagrams, labels):
    if diagram is None:
        print(f"Skipping failed parse: {sentence}")
        continue

    print(sentence)
    # print("diagram cod:", diagram.cod, " | cod length:", len(diagram.cod))

    try:
        # diagram = rewriter(diagram)
        # diagram = remove_cups(diagram)
        diagram = diagram.normal_form()
        diagram = diagram.pregroup_normal_form()
        # diagram = unify(diagram)
        # diagram = diagram.normal_form()
        
        # print("after remove_cups cod:", diagram.cod, " | cod length:", len(diagram.cod))
        # print()

        circuit = ansatz(diagram)

        circuits.append(circuit)
        kept_sentences.append(sentence)
        kept_labels.append(label)

    except Exception as e:
        print(f"Skipping sentence due to diagram/circuit error: {sentence}")
        print(type(e).__name__, e)

kept_labels = np.array(kept_labels)

for sentence, circuit in zip(kept_sentences, circuits):
    tk_circuit = circuit.to_tk()
    # print(sentence)
    print("qubits:", tk_circuit.n_qubits, "| gates:", tk_circuit.n_gates, "| depth:", tk_circuit.depth())
    # print()

# print(f"\nUsable circuits: {len(circuits)}")

if len(circuits) == 0:
    raise RuntimeError("No valid circuits were produced.")


model = TketModel.from_diagrams(
    circuits,
    backend_config=backend_config,
)

model.initialise_weights()
outputs = model(circuits)
# print("\nRaw model outputs:")
print(outputs)


# 0 : 100%|██████████| 4/4 [00:00<00:00, 1329.20it/s]


The company announced a new quantum computing platform.
The weather was sunny during the event.
The platform could improve language processing.
The audience applauded at the end of the presentation.
qubits: 15 | gates: 67 | depth: 7
qubits: 17 | gates: 78 | depth: 9
qubits: 13 | gates: 59 | depth: 8
qubits: 19 | gates: 87 | depth: 9
[[0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]


In [4]:
################################################
#### Consts for performance and reusability ####
################################################


# 2) Configure the raw AerSimulator for GPU + cuQuantum acceleration
simulator = AerSimulator(
    method="statevector", device="GPU",
    precision="single",
    cuStateVec_enable=True,
)

backend = AerBackend(simulation_method="statevector")
backend._qiskit_backend = simulator

comp_pass = backend.default_compilation_pass(2)

print("Available devices:", simulator.available_devices())
print("AerSimulator available devices:", AerSimulator().available_devices())
print("Backend available devices:", AerBackend()._qiskit_backend.available_devices())

Available devices: ['GPU']
AerSimulator available devices: ('CPU', 'GPU')
Backend available devices: ('CPU', 'GPU')


/home/green/QNLPModelTraining/qnlp_3_10/lib/python3.10/site-packages/pytket/extensions/qiskit/backends/aer.py:129: UserWarning: More than one backend with name 'aer_simulator' is available. Picking one.
  warnings.warn(


In [4]:
#############################################
# 1. Data Loading Functions
#############################################

def flatten(encoded: List[Dict]) -> Tuple[List, List]:
    circuits, labels = [], []
    for dct in encoded:
        circuits.extend(dct['circuits'])
        labels  .extend(dct['labels'])
    return circuits, labels


def load_encoded_data(file_path: str, amount: int = None):
    with open(file_path, "rb") as file:
        loaded_data = pickle.load(file)

    return loaded_data

# ds = load_encoded_data('Dataset/Encoded/cnn_dailymail/mem_test_ds.pkl')
# ds = load_encoded_data('Dataset/Encoded/cnn_dailymail/mem_test_ds_vol2.pkl')
ds = load_encoded_data('Dataset/Encoded/cnn_dailymail/mem_test_ds_vol3.pkl')
# ds = load_encoded_data('Dataset/Encoded/cnn_dailymail/mem_test_ds_vol4.pkl')

In [7]:
ds = load_encoded_data('Dataset/Encoded/cnn_dailymail/mem_test_ds_vol3.pkl')
dd = ds["encoded_dataset"]
ds_val = dd[0:1]
ds_test = dd[1:2]

In [8]:
ds = load_encoded_data('Dataset/Encoded/cnn_dailymail/mem_test_ds.pkl')
dd = ds["encoded_dataset"]
ds_train = dd


In [9]:
amount = 0
for idx, d in enumerate(ds_train):
    length = len(d["circuits"])
    amount += length
    print(idx, ":", length)
print(amount)
print(len(ds_val[0]["circuits"]))
print(len(ds_test[0]["circuits"]))

# 0 : 262, 1 : 349, 2 : 282 | 893
# 0 : 288, 1 : 387, 2 : 307 | VOL 2 982
# | VOL 3 898
# | VOL 4 909

0 : 262
1 : 349
2 : 282
893
58
54


In [6]:
def set_stage_configs(shots, epochs, use_stage_configs: bool = False):
    if use_stage_configs:
        stage_configs = []
        for s, a in [(2048, 0.1), (4096, 0.05), (8192, 0.02)]:
            stage_configs.append({
                'epochs': epochs,
                'shots': s,
                'optimizer_hparams': {
                    'a': a,
                    'c': 0.06 if s<=4096 else 0.02,
                    'A': 0.2 * epochs
                }
            })
    else:
        # Single “stage” using default parameters
        stage_configs = [{
            'epochs': epochs,
            'shots': shots,
            'optimizer_hparams': { 'a': 0.1, 'c': 0.06, 'A': 0.2 * epochs }
        }]

    return stage_configs

C. Practical recommendation for your setup

Given:

AerSimulator
long training time (~40s per epoch chunk)
instability
I would do:
Dataset
Split at article level (70/15/15)
Training
Use full dataset
Shuffle every epoch
Batch size: increase if possible
If too slow:
sample ~30–50% of data per epoch
NOT fixed subsets
Checkpoints
Save:
after each epoch
best validation loss model

In [ ]:
def train_quantum_summarizer(
    encoded_train: List[Dict],
    encoded_val:   List[Dict],
    encoded_test:  List[Dict],
    batch_size: int = 5,
    epochs:     int = 10,
    shots:      int = 8192, # 4096
    seed:       int = 42,
    checkpoint_dir:    str = 'saves/model_checkpoints',
    use_stage_configs: bool = True,
    stage_configs:     List[Dict] = None
) -> Tuple[TketModel, QuantumTrainer, Dict[str, float]]:

    os.makedirs(checkpoint_dir, exist_ok=True)

    train_circuits, train_labels = flatten(encoded_train)
    val_circuits,   val_labels   = flatten(encoded_val)
    test_circuits,  test_labels  = flatten(encoded_test)

    if stage_configs is None:
        stage_configs = set_stage_configs(shots, epochs, use_stage_configs)


    final_model   = None
    final_trainer = None
    best_val_acc  = -np.inf
    best_ckpt     = None

    # common metric & datasets
    acc_fn     = lambda y_hat, y: np.mean(np.argmax(y_hat,1) == np.argmax(y,1))
    eval_funcs = {'accuracy': acc_fn}
    val_ds     = Dataset(val_circuits, val_labels, shuffle=False)
    test_ds    = Dataset(test_circuits, test_labels, shuffle=False)


    for idx, cfg in enumerate(stage_configs, start=1):
        t_epochs  = cfg['epochs']
        t_shots   = cfg['shots']
        opt_hp  = cfg['optimizer_hparams']
        batch   = cfg.get('batch_size', batch_size)
        t_seed    = cfg.get('seed', seed + idx)

        print(f"\n=== Stage {idx}: epochs={t_epochs}, shots={t_shots}, batch={batch}, seed={t_seed} ===")

        backend_config = {
            'backend':     backend,
            'compilation': comp_pass,
            'shots':       t_shots,
        }

        ckpt_path = os.path.join(checkpoint_dir, f'model_stage{idx}.lt')

        if final_model is None:
            # First stage: build from scratch
            diagrams = train_circuits + val_circuits
            model = TketModel.from_diagrams(diagrams, backend_config=backend_config)
            model.initialise_weights()
        else:
            # Subsequent stages: resume from last checkpoint
            model = TketModel.from_checkpoint(best_ckpt, backend_config=backend_config)


        trainer = QuantumTrainer(
            model,
            loss_function      = BinaryCrossEntropyLoss(),
            optimizer          = SPSAOptimizer,
            optim_hyperparams  = opt_hp,
            evaluate_functions = eval_funcs,
            evaluate_on_train  = True,
            epochs             = t_epochs,
            seed               = t_seed,
            verbose            = 'text'
        )


        # Fit with early stopping
        train_ds = Dataset(train_circuits, train_labels, batch_size=batch, shuffle=True)
        hist = trainer.fit(
            train_ds,
            val_ds,
            early_stopping_criterion = 'accuracy',
            early_stopping_interval  = 3,
            minimize_criterion       = False
        )

        # Save checkpoint
        model.save(ckpt_path)
        print(f"» Saved checkpoint: {ckpt_path}")

        # 5) Track best val accuracy
        #    hist.val_metrics is a list of dicts of per-epoch val metrics
        #    (you may need to inspect trainer.history if API differs)
        final_val_acc = 0.1
        # final_val_acc = hist.val_metrics[-1]['accuracy']
        if final_val_acc > best_val_acc:
            best_val_acc = final_val_acc
            best_ckpt    = ckpt_path

        # Prepare for next stage
        final_model   = model
        final_trainer = trainer

    test_ds = Dataset(test_circuits, test_labels, shuffle=False)
    test_metrics = final_trainer.evaluate(test_ds)
    print(f"\nFinal test metrics: {test_metrics}")

    return final_model, final_trainer, test_metrics

In [13]:
# Debugging parameters:
model, trainer, metrics = train_quantum_summarizer(
  encoded_train   = ds_test,
  encoded_val     = ds_val,
  encoded_test    = ds_test,

  batch_size = 4,
  epochs     = 1,
  shots      = 32768,
  seed       = 42,
  checkpoint_dir = 'saves/debugging_model_checkpoints',
  use_stage_configs = False
)

# batch_size = 5, epochs = 1, shots = 32, seed = 42,
# train/time: 2m36s   train/time_per_epoch: 2m36s   train/time_per_step: 2.95s   valid/time: 1m16s   valid/time_per_eval: 1m16s

# epochs=1, shots=4, batch=5, seed=42
# train/time: 2m33s   train/time_per_epoch: 2m33s   train/time_per_step: 2.88s   valid/time: 1m15s   valid/time_per_eval: 1m15s

# epochs=1, shots=4, batch=64, seed=42
# train/time: 2m30s   train/time_per_epoch: 2m30s   train/time_per_step: 29.99s   valid/time: 1m14s   valid/time_per_eval: 1m14s

# epochs=1, shots=1, batch=128, seed=42
# train/time: 2m31s   train/time_per_epoch: 2m31s   train/time_per_step: 50.34s   valid/time: 1m14s   valid/time_per_eval: 1m14s




=== Stage 1: epochs=1, shots=32768, batch=4, seed=43 ===
» Saved checkpoint: saves/debugging_model_checkpoints/model_stage1.lt


Epoch 1:  train/loss: 1.4093   valid/loss: 1.1038   train/time: 37.07s   valid/time: 18.83s   train/accuracy: 0.8333   valid/accuracy: 0.8448

Training completed!
train/time: 37.07s   train/time_per_epoch: 37.07s   train/time_per_step: 2.65s   valid/time: 18.83s   valid/time_per_eval: 18.83s


AttributeError: 'QuantumTrainer' object has no attribute 'evaluate'

In [63]:
# Custom 2-stage strategy:
stages = [
  {'epochs': 5,  'shots': 1024,
   'optimizer_hparams': {'a':0.15,'c':0.06,'A':0.2*5},
   'batch_size':  8},
  {'epochs': 15, 'shots': 8192,
   'optimizer_hparams': {'a':0.02,'c':0.02,'A':0.2*15}}
]

model, trainer, metrics = train_quantum_summarizer(
  encoded_train   = ds_test,
  encoded_val     = ds_val,
  encoded_test    = ds_test,

  batch_size = 5,
  seed       = 123,
  # epochs = 10,
  # shots = 8192,
  stage_configs   = stages
)




=== Stage 1: epochs=5, shots=1024, batch=8, seed=124 ===


Epoch 1:  train/loss: 2.4439   valid/loss: 2.8064   train/time: 37.81s   valid/time: 18.21s   train/accuracy: 0.7407   valid/accuracy: 0.7069
Epoch 2:  train/loss: 2.4041   valid/loss: 2.5506   train/time: 38.31s   valid/time: 18.53s   train/accuracy: 0.7963   valid/accuracy: 0.6552
Epoch 3:  train/loss: 0.6235   valid/loss: 1.4374   train/time: 37.50s   valid/time: 18.58s   train/accuracy: 0.7778   valid/accuracy: 0.7586
Epoch 4:  train/loss: 3.9267   valid/loss: 1.3794   train/time: 38.26s   valid/time: 17.26s   train/accuracy: 0.7963   valid/accuracy: 0.8103
Epoch 5:  train/loss: 0.9845   valid/loss: 0.8217   train/time: 36.65s   valid/time: 20.39s   train/accuracy: 0.7593   valid/accuracy: 0.7414

Training completed!
train/time: 3m9s   train/time_per_epoch: 37.71s   train/time_per_step: 5.39s   valid/time: 1m33s   valid/time_per_eval: 18.59s


» Saved checkpoint: saves/model_checkpoints/model_stage1.lt

=== Stage 2: epochs=15, shots=8192, batch=5, seed=125 ===


Epoch 1:   train/loss: 5.5274   valid/loss: 2.1232   train/time: 35.30s   valid/time: 19.44s   train/accuracy: 0.8333   valid/accuracy: 0.7931
Epoch 2:   train/loss: 0.8013   valid/loss: 2.4750   train/time: 35.14s   valid/time: 18.43s   train/accuracy: 0.7407   valid/accuracy: 0.7241
Epoch 3:   train/loss: 5.5922   valid/loss: 2.4570   train/time: 35.00s   valid/time: 17.84s   train/accuracy: 0.8333   valid/accuracy: 0.7069
Epoch 4:   train/loss: 3.1757   valid/loss: 1.3858   train/time: 36.24s   valid/time: 16.85s   train/accuracy: 0.8333   valid/accuracy: 0.8448
Epoch 5:   train/loss: 3.2208   valid/loss: 1.7549   train/time: 36.98s   valid/time: 17.67s   train/accuracy: 0.8704   valid/accuracy: 0.8103
Epoch 6:   train/loss: 3.0971   valid/loss: 1.8732   train/time: 36.44s   valid/time: 17.97s   train/accuracy: 0.7963   valid/accuracy: 0.7069


» Saved checkpoint: saves/model_checkpoints/model_stage2.lt


Epoch 7:   train/loss: 0.6065   valid/loss: 1.4815   train/time: 36.06s   valid/time: 16.87s   train/accuracy: 0.8148   valid/accuracy: 0.7414
Early stopping!
Best model (epoch=4, step=44) saved to
runs/Apr07_19-42-53_DESKTOP-5DCJBRK/best_model.lt

Training completed!
train/time: 4m11s   train/time_per_epoch: 35.88s   train/time_per_step: 3.26s   valid/time: 2m5s   valid/time_per_eval: 17.87s


AttributeError: 'QuantumTrainer' object has no attribute 'evaluate'